In [19]:
from exchangelib import Credentials, Account, Configuration, DELEGATE
import re
import webbrowser

from selenium import webdriver 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By

import os
import zipfile

import time

# Set your email, password, and the Exchange server details
email_address = 'keitaro4@outlook.com'
password = 'Michiko325'
server = 'outlook.office365.com'  # Common for Office 365 accounts

# Set credentials and configure the account
credentials = Credentials(email_address, password)
config = Configuration(server=server, credentials=credentials)
account = Account(primary_smtp_address=email_address, config=config, autodiscover=False, access_type=DELEGATE)

# Define your search criteria
sender_email = 'metro_tokyo@saas.elib.gprime.jp'

# Access the "TokyoArchives" folder
tokyo_archives = None
for folder in account.root.walk():
    if folder.name == 'TokyoArchives':
        tokyo_archives = folder
        break

if tokyo_archives is None:
    print("TokyoArchives folder not found.")
else:
    # Fetch a certain number of recent emails
    num_emails_to_check = 100
    emails = tokyo_archives.all().order_by('-datetime_received')[:num_emails_to_check]

    # Iterate through the emails
    for email in emails:
        # Check if the body is of type HTML or plain text
        if email.body.body_type == 'HTML':
            body = str(email.body)
        else:
            body = email.text_body

        # Sample email content
        email_content = body

        # Regular expressions to extract URL, ID, and password
        url_pattern = r'画像ダウンロードURL：\s*(https?://\S+)'
        id_pattern = r'認証ID\s*:\s*(\S+)'
        password_pattern = r'認証パスワード\s*：\s*(\S+)'

        # Find matches in the email content
        url_match = re.search(url_pattern, email_content)
        id_match = re.search(id_pattern, email_content)
        password_match = re.search(password_pattern, email_content)

        # Extract the URL, ID, and password
        url = url_match.group(1) if url_match else None
        your_id = id_match.group(1) if id_match else None
        your_password = password_match.group(1) if password_match else None

        # Ensure all information was found
        if not all([url, your_id, your_password]):
            print("Failed to extract all necessary information from the email.")
        else:
            # Set up the Selenium WebDriver
            options = webdriver.ChromeOptions()
            options.add_argument('--port=12345')  # Specify a different port number

            # Create a new instance of the Chrome WebDriver
            driver = webdriver.Chrome(options=options)

            # Open the URL
            driver.get(url)

            # Wait for the page to load
            driver.implicitly_wait(10)

            # Find the ID and password input fields and the button using the new method
            id_input = driver.find_element(By.XPATH, '/html/body/form/div[1]/div[1]/div[1]/input')
            password_input = driver.find_element(By.XPATH, '/html/body/form/div[1]/div[1]/div[2]/input')
            submit_button = driver.find_element(By.XPATH, '//*[@id="command"]/div[1]/div[2]/input[1]')

            # Enter the ID and password
            id_input.send_keys(your_id)
            password_input.send_keys(your_password)

            # Click the button
            submit_button.click()
            
            # Set your download directory and expected file pattern
            download_dir = "C:/Users/Keitaro Ninomiya/Downloads"
            expected_file_pattern = r'\d{11}\.zip$'  # Adjust this to match the expected file

            # Function to check if the file exists
            def file_exists_in_directory(directory, pattern):
                for filename in os.listdir(directory):
                    if pattern in filename:
                        return True
                return False

            # Wait for the file to appear in the directory
            max_wait = 30  # seconds to wait
            wait_interval = 5  # seconds between checks
            elapsed_time = 0

            while elapsed_time < max_wait:
                if file_exists_in_directory(download_dir, expected_file_pattern):
                    print("File has been downloaded.")
                    break
                time.sleep(wait_interval)
                elapsed_time += wait_interval

            if elapsed_time >= max_wait:
                print("Timed out waiting for the file to download.")

            # Function to check if the file exists
            def file_exists_in_directory(directory, pattern):
                for filename in os.listdir(directory):
                    if pattern in filename:
                        return True
                return False

            # Wait for download
            driver.implicitly_wait(30)

            # Set the path to the directory containing the downloaded ZIP file
            download_dir = "C:/Users/Keitaro Ninomiya/Downloads"

            # Set the directory where you want to extract the images
            extract_to_dir = 'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Data_Collection/Download_Image/ExtractedImage'
            # Ensure the output directory exists
            os.makedirs(extract_to_dir, exist_ok=True)
            zip_pattern = r'\d{11}\.zip$'

            def find_most_recent_zip(directory, pattern):
                zip_files = [f for f in os.listdir(directory) if re.match(pattern, f)]
                if not zip_files:
                    return None
                latest_file = max(zip_files, key=lambda x: os.path.getctime(os.path.join(directory, x)))
                return latest_file
            
            # Find the most recent ZIP file in the download directory
            zip_file_name = find_most_recent_zip(download_dir, zip_pattern)

            if zip_file_name:
                zip_file_path = os.path.join(download_dir, zip_file_name)

                # Extract the ZIP file
                with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_to_dir)

                print(f"Images extracted from {zip_file_name} to {extract_to_dir}")
            else:
                print("No matching ZIP file found.")
                        


Timed out waiting for the file to download.
Images extracted from 20231221023.zip to C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Data_Collection/Download_Image/ExtractedImage
Timed out waiting for the file to download.
Images extracted from 20231221021.zip to C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Data_Collection/Download_Image/ExtractedImage
Timed out waiting for the file to download.
Images extracted from 20231221024.zip to C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Data_Collection/Download_Image/ExtractedImage
Timed out waiting for the file to download.
Images extracted from 20231221025.zip to C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Data_Collection/Download_Image/ExtractedImage
Timed out waiting for the file to download.
Images extracted from 20231221020.zip to C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/To